<a href="https://colab.research.google.com/github/hariharansg01-hub/NutriGuide_AI_Agent/blob/main/NutriGuide_AI_MCP_Client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install "mcp[cli]" groq pandas

In [17]:
import pandas as pd

food_data = {
    "food": ["boiled egg", "milk", "banana", "rice", "chicken breast"],
    "serving_g": [50, 100, 100, 100, 100],
    "calories": [78, 61, 89, 130, 165],
    "protein": [6.3, 3.2, 1.1, 2.7, 31]
}

food_df = pd.DataFrame(food_data)

food_df

,food,serving_g,calories,protein
0,boiled egg,50,78,6.3
1,milk,100,61,3.2
2,banana,100,89,1.1
3,rice,100,130,2.7
4,chicken breast,100,165,31.0


In [18]:
from mcp.server import MCPServer

# Create the MCP server
mcp_server = MCPServer("NutriGuide Nutrition Server")


# Tool 1: Get calories
@mcp_server.tool()
def get_calories(food_name: str) -> str:
    """Get calorie information for a food item per 100g."""

    result = food_df[
        food_df["food"].str.lower() == food_name.lower()
    ]

    if result.empty:
        return f"{food_name} not found in the nutrition database."

    calories = result.iloc[0]["calories"]

    return f"{food_name} contains {calories} kcal per 100g."


# Tool 2: Get protein
@mcp_server.tool()
def get_protein(food_name: str) -> str:
    """Get protein information for a food item per 100g."""

    result = food_df[
        food_df["food"].str.lower() == food_name.lower()
    ]

    if result.empty:
        return f"{food_name} not found in the nutrition database."

    protein = result.iloc[0]["protein"]

    return f"{food_name} contains {protein}g protein per 100g."


print("NutriGuide MCP server created!")

NutriGuide MCP server created!


In [19]:
import asyncio

async def check_mcp_tools():

    available_tools = await mcp_server.list_tools()

    print("Tools registered in NutriGuide MCP Server:\n")

    for tool in available_tools:
        print("Tool name:", tool.name)
        print("Description:", tool.description)
        print("-" * 40)

await check_mcp_tools()

Tools registered in NutriGuide MCP Server:

Tool name: get_calories
Description: Get calorie information for a food item per 100g.
----------------------------------------
Tool name: get_protein
Description: Get protein information for a food item per 100g.
----------------------------------------


In [20]:
from mcp import Client

async def connect_to_nutrition_server():

    # Connect MCP client to our server
    async with Client(mcp_server) as client:

        print("Connected to NutriGuide MCP Server!")

        # Discover available tools
        result = await client.list_tools()

        print("\nTools discovered by MCP Client:\n")

        for tool in result.tools:
            print("Tool:", tool.name)
            print("Description:", tool.description)
            print("-" * 40)

await connect_to_nutrition_server()

Connected to NutriGuide MCP Server!

Tools discovered by MCP Client:

Tool: get_calories
Description: Get calorie information for a food item per 100g.
----------------------------------------
Tool: get_protein
Description: Get protein information for a food item per 100g.
----------------------------------------


In [21]:
async def test_calorie_tool():

    async with Client(mcp_server) as client:

        result = await client.call_tool(
            "get_calories",
            {"food_name": "chicken breast"}
        )

        print("MCP Server Response:")

        for item in result.content:
            if hasattr(item, "text"):
                print(item.text)


await test_calorie_tool()

MCP Server Response:
chicken breast contains 165 kcal per 100g.


In [22]:
async def test_protein_tool():

    async with Client(mcp_server) as client:

        result = await client.call_tool(
            "get_protein",
            {"food_name": "chicken breast"}
        )

        print("MCP Server Response:")

        for item in result.content:
            if hasattr(item, "text"):
                print(item.text)


await test_protein_tool()

MCP Server Response:
chicken breast contains 31.0g protein per 100g.


In [35]:
from groq import Groq
from google.colab import userdata

groq_client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

MODEL = "openai/gpt-oss-120b"

print("NutriGuide AI Agent connected to Groq!")

NutriGuide AI Agent connected to Groq!


In [36]:
async def discover_tools_for_ai():

    async with Client(mcp_server) as mcp_client:

        result = await mcp_client.list_tools()

        groq_tools = []

        for tool in result.tools:

            groq_tools.append({
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.input_schema
                }
            })

        return groq_tools


tools_for_ai = await discover_tools_for_ai()

print("Tools available to NutriGuide AI:\n")

for tool in tools_for_ai:
    print(tool["function"]["name"])
    print(tool["function"]["description"])
    print("-" * 40)

Tools available to NutriGuide AI:

get_calories
Get calorie information for a food item per 100g.
----------------------------------------
get_protein
Get protein information for a food item per 100g.
----------------------------------------


In [37]:
messages = [
    {
        "role": "system",
        "content": """
        You are NutriGuide AI, a nutrition assistant.
        Use the available tools whenever the user asks
        for calorie or protein information.
        Never invent nutrition values.
        """
    },
    {
        "role": "user",
        "content": "How many calories are in 100g of chicken breast?"
    }
]

response = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools_for_ai,
    tool_choice="auto"
)

message = response.choices[0].message

print("AI response:", message.content)
print("Tool calls:", message.tool_calls)

AI response: None
Tool calls: [ChatCompletionMessageToolCall(id='fc_50dcfaac-4425-4072-9ae8-2f8c93395e75', function=Function(arguments='{"food_name":"chicken breast"}', name='get_calories'), type='function')]


In [38]:
async def execute_mcp_tool(tool_call):

    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    async with Client(mcp_server) as mcp_client:

        result = await mcp_client.call_tool(
            function_name,
            arguments
        )

        return "\n".join(
            item.text
            for item in result.content
            if hasattr(item, "text")
        )


import json

for tool_call in message.tool_calls:

    print("Executing MCP tool:", tool_call.function.name)

    tool_result = await execute_mcp_tool(tool_call)

    print("MCP Server Result:", tool_result)

Executing MCP tool: get_calories
MCP Server Result: chicken breast contains 165 kcal per 100g.


In [39]:
# Send the tool result back to Groq

messages.append(message)

for tool_call in message.tool_calls:

    tool_result = await execute_mcp_tool(tool_call)

    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": tool_result
    })


# Ask Groq to generate the final answer

final_response = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages
)

ai_answer = final_response.choices[0].message.content

messages.append({
    "role": "assistant",
    "content": ai_answer
})

print("NutriGuide AI:", ai_answer)

NutriGuide AI: Chicken breast provides about **165 kcal per 100 g**.


In [32]:
# Start a fresh conversation
agent_messages = [
    {
        "role": "system",
        "content": """
        You are NutriGuide AI, a nutrition assistant.

        1. Use MCP tools for calorie and protein questions.
        2. Never invent nutrition values.
        3. Use the results returned by MCP tools.
        4. Give clear and friendly answers.
        """
    }
]


async def chat_with_agent(user_input):

    # Add user's question
    agent_messages.append({
        "role": "user",
        "content": user_input
    })

    # Ask Groq to select an appropriate tool
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=agent_messages,
        tools=tools_for_ai,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # Check whether Groq requested a tool
    if message.tool_calls:

        agent_messages.append(message)

        # Execute each requested MCP tool
        for tool_call in message.tool_calls:

            print("MCP Tool Selected:", tool_call.function.name)

            result = await execute_mcp_tool(tool_call)

            print("MCP Server Result:", result)

            agent_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        # Generate the final AI response
        final_response = groq_client.chat.completions.create(
            model=MODEL,
            messages=agent_messages
        )

        answer = final_response.choices[0].message.content

    else:
        answer = message.content

    agent_messages.append({
        "role": "assistant",
        "content": answer
    })

    return answer


print("NutriGuide AI Agent is ready!")

NutriGuide AI Agent is ready!


In [33]:
import asyncio

print(await chat_with_agent(
    "How many calories are in 100g of chicken breast?"
))

MCP Tool Selected: get_calories
MCP Server Result: chicken breast contains 165 kcal per 100g.
Chicken breast contains **about 165 kcal per 100 g**.


In [34]:
print(await chat_with_agent(
    "How much protein is present in 100g of chicken breast?"
))

MCP Tool Selected: get_protein
MCP Server Result: chicken breast contains 31.0g protein per 100g.
Chicken breast provides **approximately 31 g of protein per 100 g**.
